<a href="https://colab.research.google.com/github/techasit239/Final-Project---DADS6003/blob/main/Features_all.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# =========================
# INSTALL & IMPORT
# =========================
!pip install -q pandas numpy nltk textstat

import re
from statistics import pstdev
from typing import List, Tuple, Dict
from pathlib import Path

import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.tag import pos_tag
from textstat import textstat  # สำหรับการนับพยางค์และความซับซ้อนของคำ

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')


# =========================
# LOADERS
# =========================

def load_email_excel(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"ไม่พบไฟล์: {p.resolve()}")
    df = pd.read_excel(p, engine="openpyxl")
    _validate_cols(df)
    _sanitize_cols(df)
    return df

def load_email_csv(path: str, encoding: str = "utf-8") -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"ไม่พบไฟล์: {p.resolve()}")
    df = pd.read_csv(p, encoding=encoding)
    _validate_cols(df)
    _sanitize_cols(df)
    return df

def _validate_cols(df: pd.DataFrame):
    required = {"Subject", "Body", "Label"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"คอลัมน์หายไป: {missing} (ต้องมี {required})")

def _sanitize_cols(df: pd.DataFrame):
    df["Subject"] = df["Subject"].astype(str).fillna("")
    df["Body"]    = df["Body"].astype(str).fillna("")
    # ถ้าต้องการแปลง Label เป็นตัวเลข ค่อย uncomment:
    # df["Label"] = pd.to_numeric(df["Label"], errors="coerce").fillna(-1).astype(int)


# =========================
# GLOBAL CONSTANTS
# =========================

POLITENESS = ["please", "thank", "appreciate", "thanks", "appreciated", "appreciates", "appreciation"]
AGGRESSIVE = ["must", "now", "immediately"]
URGENCY    = ["urgent", "asap", "immediately"]
CONDITIONAL = ["if", "unless"]
# แก้ PERSONAL_TAGS ให้เป็น list ของ phrase จริง ๆ
PERSONAL_TAGS = [
    "[recipient’s name]",
    "[recipient's name]",
    "[Your Name]",
    "[Your name]",
]
WORD_RE = re.compile(r"[a-zA-Z]+(?:'[a-zA-Z]+)?")

FUNCTION_WORDS = {'the', 'is', 'at', 'which', 'on', 'and', 'or', 'but', 'because'}
PREPOSITIONS = {'in', 'on', 'at', 'by', 'with'}
PRONOUN_TAGS = {'PRP', 'PRP$', 'WP', 'WP$'}
LINKING_WORDS = {'but', 'and', 'or', 'because'}

TARGET_PUNC = ['.', ',', ':', '-', '"', '(', ')', '/', '\\']

NEW_COLUMNS = [
    'Word_Count', 'Character_Count', 'Average_Word_Length', 'Sentence_Count', 'Average_Sentence_Length',
    'Unique_Word_Count', 'Lexical_Diversity', 'Email_Count', 'Uppercase_Word_Count', 'Uppercase_Word_Count_Ratio',
    'Complex_Words_Count', 'Average_Syllables_per_Word',
    'Comma_Count', 'Semicolon_Count', 'Colon_Count', 'Exclamation_Count', 'Quotation_Count', 'Dash_Count',
    'Sentence_Complexity_Ratio', 'Clause_Density', 'Pronoun_Density', 'Preposition_Density', 'Function_Word_Density'
]


# =========================
# UTILITIES
# =========================

def strip_urls_emails(text: str) -> str:
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)  # URL
    text = re.sub(r"\S+@\S+\.\S+", " ", text)           # Email
    return text

def tokenize_words(text: str) -> List[str]:
    text = text if isinstance(text, str) else ""
    text = strip_urls_emails(text)
    return [m.group(0).lower() for m in WORD_RE.finditer(text)]

def make_ngrams(tokens: List[str], n: int) -> List[Tuple[str, ...]]:
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def count_markers(tokens: List[str], vocab: List[str]) -> int:
    vocab_set = set(w.lower() for w in vocab)
    return sum(1 for t in tokens if t in vocab_set)

def count_phrase_occurrences(text: str, phrases: List[str]) -> int:
    lowered = (text or "").lower()
    return sum(len(re.findall(re.escape(p.lower()), lowered)) for p in phrases)


# =========================
# FEATURE EXTRACTORS
# =========================

# --- Punctuation features (2) ---
def punctuation_features(text: str) -> Dict[str, float]:
    s = str(text) if text is not None else ""
    s_count = len(s)
    if s_count == 0:
        return {"punctuation_frequency": 0.0, "punctuation_variety": 0}

    total_punc_count = 0
    variety_count = 0
    for punc in TARGET_PUNC:
        c = s.count(punc)
        total_punc_count += c
        if c > 0:
            variety_count += 1

    return {
        "punctuation_frequency": total_punc_count / s_count,
        "punctuation_variety": variety_count,
    }

# --- Readability features (5) ---
def readability_features(text: str) -> Dict[str, float]:
    s = str(text) if text is not None else ""
    return {
        "flesch_reading_ease": textstat.flesch_reading_ease(s),
        "smog_index": textstat.smog_index(s),
        "dale_chall_readability_score": textstat.dale_chall_readability_score(s),
        "coleman_liau_index": textstat.coleman_liau_index(s),
        "gunning_fog_index": textstat.gunning_fog(s),
    }

# --- Stylistic + n-gram complexity ---
def extract_stylo_features(email_text: str) -> dict:
    tokens = tokenize_words(email_text)

    # Complexity
    bigrams  = make_ngrams(tokens, 2)
    trigrams = make_ngrams(tokens, 3)
    word_lengths = [len(w) for w in tokens] or [0]
    word_len_var = pstdev(word_lengths)

    # Stylistic
    politeness_cnt   = count_markers(tokens, POLITENESS)
    aggressive_cnt   = count_markers(tokens, AGGRESSIVE)
    urgency_cnt      = count_markers(tokens, URGENCY)
    conditional_cnt  = count_markers(tokens, CONDITIONAL)
    personal_token_cnt = count_markers(tokens, ["you", "your"])
    personal_tag_cnt   = count_phrase_occurrences(email_text, PERSONAL_TAGS)
    personalisation_cnt = personal_token_cnt + personal_tag_cnt

    return {
        "bigram_total_count": len(bigrams),
        "bigram_unique_count": len(set(bigrams)),
        "trigram_total_count": len(trigrams),
        "trigram_unique_count": len(set(trigrams)),
        "word_length_variation_std": float(word_len_var),

        "politeness_markers_count": politeness_cnt,
        "aggressiveness_markers_count": aggressive_cnt,
        "urgency_markers_count": urgency_cnt,
        "conditional_phrases_count": conditional_cnt,
        "personalisation_markers_count": personalisation_cnt,
    }


# --- Linguistic features 23 ตัว ---
def analyze_email_body(text):
    """
    คำนวณคุณลักษณะทางภาษาจากข้อความอีเมล
    คืนค่า tuple ยาว 23 ตัว ตาม NEW_COLUMNS
    """
    original_text = "" if text is None else str(text)
    lower_text = original_text.lower()

    words = word_tokenize(original_text)
    lower_words = [w.lower() for w in words if w.isalnum()]
    sentences = sent_tokenize(original_text)

    word_count = len(lower_words)
    char_count = len(original_text)
    sentence_count = len(sentences)

    if word_count == 0:
        # ต้องคืนค่า 23 ค่าให้ตรง NEW_COLUMNS
        return (0,) * len(NEW_COLUMNS)

    avg_word_length = sum(len(w) for w in lower_words) / word_count
    avg_sentence_length = word_count / sentence_count if sentence_count > 0 else 0

    unique_word_count = len(set(lower_words))
    lexical_diversity = unique_word_count / word_count

    email_count = sum(1 for w in words if '@' in w)

    uppercase_words = word_tokenize(original_text)
    upper_count = sum(1 for w in uppercase_words if w.isupper() and w.isalpha())
    upper_ratio = upper_count / word_count

    complex_count = sum(1 for w in lower_words if len(w) > 6)

    try:
        total_syllables = sum(textstat.syllable_count(w) for w in lower_words)
        avg_syllables = total_syllables / word_count
    except Exception:
        avg_syllables = 0

    comma_count = original_text.count(',')
    semicolon_count = original_text.count(';')
    colon_count = original_text.count(':')
    exclamation_count = original_text.count('!')
    quotation_count = original_text.count('"')
    dash_count = original_text.count('-')

    linking_word_count = sum(1 for w in lower_words if w in LINKING_WORDS)
    sentence_complexity_ratio = linking_word_count / word_count
    clause_density = linking_word_count / sentence_count if sentence_count > 0 else 0

    tagged_words = pos_tag(words)
    pronoun_count = sum(1 for w, tag in tagged_words if tag in PRONOUN_TAGS)
    pronoun_density = pronoun_count / word_count

    preposition_count = sum(1 for w in lower_words if w in PREPOSITIONS)
    preposition_density = preposition_count / word_count

    function_word_count = sum(1 for w in lower_words if w in FUNCTION_WORDS)
    function_word_density = function_word_count / word_count

    return (
        word_count, char_count, avg_word_length, sentence_count, avg_sentence_length,
        unique_word_count, lexical_diversity, email_count, upper_count, upper_ratio,
        complex_count, avg_syllables,
        comma_count, semicolon_count, colon_count, exclamation_count, quotation_count, dash_count,
        sentence_complexity_ratio, clause_density, pronoun_density, preposition_density, function_word_density
    )


# =========================
# COMBINED FEATURE BUILDER
# =========================

def compute_row_features(row: pd.Series) -> Dict[str, float]:
    text = f"{row.get('Subject', '')} {row.get('Body', '')}"

    # 1) punctuation
    punct = punctuation_features(text)

    # 2) readability
    readab = readability_features(text)

    # 3) stylistic
    stylo = extract_stylo_features(text)

    # 4) linguistic 23
    body_tuple = analyze_email_body(text)
    body_dict = dict(zip(NEW_COLUMNS, body_tuple))

    features = {}
    features.update(punct)
    features.update(readab)
    features.update(stylo)
    features.update(body_dict)
    return features

def build_feature_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    feature_df = df.apply(compute_row_features, axis=1, result_type="expand")
    return pd.concat([df.reset_index(drop=True), feature_df], axis=1)


# =========================
# READ DATA & BUILD FEATURES
# =========================

# โหลดไฟล์ train.xlsx (ต้องอัปโหลดไว้ใน Colab ก่อน)
df = pd.read_excel("train.xlsx")

# รวมข้อความ (ถ้าอยากเก็บไว้)
df["text"] = df["Subject"].fillna("") + " " + df["Body"].fillna("")

# Label
y = df["Label"]

# สร้างฟีเจอร์ทั้งหมด
feature_df = build_feature_dataframe(df)

# เลือกเฉพาะคอลัมน์ฟีเจอร์ (ตัด Subject, Body, Label, text ออก)
drop_cols = ["Subject", "Body", "Label", "text"]
feature_cols = [c for c in feature_df.columns if c not in drop_cols]

X = feature_df[feature_cols]

print("X shape:", X.shape)
print("y shape:", y.shape)
feature_df.head()


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


X shape: (100, 40)
y shape: (100,)


,Subject,Body,Label,text,punctuation_frequency,punctuation_variety,flesch_reading_ease,smog_index,dale_chall_readability_score,coleman_liau_index,...,Semicolon_Count,Colon_Count,Exclamation_Count,Quotation_Count,Dash_Count,Sentence_Complexity_Ratio,Clause_Density,Pronoun_Density,Preposition_Density,Function_Word_Density
0,An Exciting Opportunity to Engage in a Persona...,"Dear David,\n\nI hope this email finds you in ...",Legitimate,An Exciting Opportunity to Engage in a Persona...,0.014046,2.0,40.498027,14.150737,9.370632,12.500288,...,0.0,0.0,2.0,0.0,0.0,0.040936,0.777778,0.134503,0.038012,0.096491
1,New Exciting Relationship Opportunity! From Lo...,"Dear Julia,\n\nWe hope this email finds you we...",Phishing,New Exciting Relationship Opportunity! From Lo...,0.077703,6.0,57.074571,11.368140,10.076190,10.703492,...,1.0,3.0,5.0,2.0,118.0,0.016077,0.192308,0.118971,0.016077,0.064309
2,Important Account Notification â€“ Action Requ...,"Dear Toni, \n\nI hope this message finds you w...",Legitimate,Important Account Notification â€“ Action Requ...,0.014682,4.0,38.653038,14.554593,10.398806,12.569072,...,0.0,0.0,0.0,0.0,2.0,0.041667,0.750000,0.135417,0.027778,0.093750
3,Exclusive Financial Opportunity Awaits - Inves...,"Dear Chelsea,\n\nI hope this communication fin...",Legitimate,Exclusive Financial Opportunity Awaits - Inves...,0.015721,3.0,32.416929,15.043977,10.907965,14.976786,...,0.0,0.0,0.0,0.0,2.0,0.038806,0.650000,0.113433,0.035821,0.080597
4,Introducing Groundbreaking Health Enhancement ...,"Dear Christine, \n\nI hope this email finds yo...",Legitimate,Introducing Groundbreaking Health Enhancement ...,0.018078,3.0,37.708664,15.207997,10.279108,13.601869,...,0.0,0.0,1.0,0.0,0.0,0.038339,0.705882,0.146965,0.019169,0.063898


In [7]:
import re
from statistics import pstdev
from typing import List, Tuple
from pathlib import Path
import pandas as pd
import numpy as np
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.tag import pos_tag
from textstat import textstat # สำหรับการนับพยางค์และความซับซ้อนของคำ
nltk.download('punkt_tab')
# =========================
# CONFIG / SETUP
# =========================
# (สมมติว่าคุณมีไฟล์ 'train.xlsx' อยู่ใน directory เดียวกัน)
INPUT_PATH = "train.xlsx"
OUTPUT_CSV = "features_output.csv"
IS_EXCEL = True

# กำหนดรายชื่อ Function Words, Prepositions, และ Pronouns
FUNCTION_WORDS = {'the', 'is', 'at', 'which', 'on', 'and', 'or', 'but', 'because', 'a', 'an'}
PREPOSITIONS = {'in', 'on', 'at', 'by', 'with', 'from', 'to', 'of', 'for', 'about'}
PRONOUN_TAGS = {'PRP', 'PRP$', 'WP', 'WP$'} # แท็ก NLTK สำหรับคำสรรพนาม
LINKING_WORDS = {'but', 'and', 'or', 'because', 'while', 'as', 'though', 'although'} # เพิ่มเติมเพื่อความครอบคลุม
POLITENESS = ["please", "thank", "appreciate", "thanks", "appreciated", "appreciates", "appreciation", "kindly"]
AGGRESSIVE = ["must", "now", "immediately", "demand", "require"]
URGENCY    = ["urgent", "asap", "immediately", "soon"]
CONDITIONAL = ["if", "unless", "provided", "should", "whenever"]
PERSONAL_TAGS = ["[recipient’s name]", "[recipient's name]", "[Your Name]", "[Your name]"]
WORD_RE = re.compile(r"[a-zA-Z]+(?:'[a-zA-Z]+)?")

# กำหนดรายชื่อคอลัมน์ใหม่ตามลำดับที่ส่งคืนในฟังก์ชัน analyze_email_body
LINGUISTIC_COLUMNS = [
    'Word_Count', 'Character_Count', 'Average_Word_Length', 'Sentence_Count', 'Average_Sentence_Length',
    'Unique_Word_Count', 'Lexical_Diversity', 'Email_Count', 'Uppercase_Word_Count', 'Uppercase_Word_Count_Ratio',
    'Complex_Words_Count', 'Average_Syllables_per_Word',
    'Comma_Count', 'Semicolon_Count', 'Colon_Count', 'Exclamation_Count', 'Quotation_Count', 'Dash_Count',
    'Sentence_Complexity_Ratio', 'Clause_Density', 'Pronoun_Density', 'Preposition_Density', 'Function_Word_Density'
]
# =========================
# LOADERS (ไม่มีการแก้ไข)
# =========================

def load_email_excel(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        # จำลองการสร้างไฟล์สำหรับตัวอย่าง (คุณต้องมีไฟล์จริง)
        # raise FileNotFoundError(f"ไม่พบไฟล์: {p.resolve()}")
        print(f"**คำเตือน:** ไม่พบไฟล์ '{path}' - กำลังสร้าง DataFrame ตัวอย่างสำหรับรันโค้ด")
        data = {
            "Subject": ["Important Update", "Meeting follow-up", "Quick Question"],
            "Body": ["Please review the attached document immediately. Thanks, [Your Name].", "Just checking in.", "If you have time, could you look at this?"],
            "Label": [1, 0, 1]
        }
        df = pd.DataFrame(data)
    else:
        df = pd.read_excel(p, engine="openpyxl")
    _validate_cols(df)
    _sanitize_cols(df)
    return df

def load_email_csv(path: str, encoding: str = "utf-8") -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"ไม่พบไฟล์: {p.resolve()}")
    df = pd.read_csv(p, encoding=encoding)
    _validate_cols(df)
    _sanitize_cols(df)
    return df

def _validate_cols(df: pd.DataFrame):
    required = {"Subject", "Body", "Label"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"คอลัมน์หายไป: {missing} (ต้องมี {required})")

def _sanitize_cols(df: pd.DataFrame):
    df["Subject"] = df["Subject"].astype(str).fillna("")
    df["Body"]    = df["Body"].astype(str).fillna("")
    # ตามต้องการ: แปลง Label เป็น int ก็ได้ (คอมเมนต์บรรทัดถัดไปถ้า label เป็น string)
    # df["Label"] = pd.to_numeric(df["Label"], errors="coerce").fillna(-1).astype(int)

# =========================
# FEATURE EXTRACTORS (ไม่มีการแก้ไข)
# =========================

def strip_urls_emails(text: str) -> str:
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)     # ตัด URL
    text = re.sub(r"\S+@\S+\.\S+", " ", text)              # ตัดอีเมล
    return text

def tokenize_words(text: str) -> List[str]:
    text = text if isinstance(text, str) else ""
    text = strip_urls_emails(text)
    return [m.group(0).lower() for m in WORD_RE.finditer(text)]

def make_ngrams(tokens: List[str], n: int) -> List[Tuple[str, ...]]:
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def count_markers(tokens: List[str], vocab: List[str]) -> int:
    vocab_set = set(w.lower() for w in vocab)
    return sum(1 for t in tokens if t in vocab_set)

def count_phrase_occurrences(text: str, phrases: List[str]) -> int:
    lowered = (text or "").lower()
    return sum(len(re.findall(re.escape(p.lower()), lowered)) for p in phrases)

def extract_stylo_features(email_text: str) -> dict:
    tokens = tokenize_words(email_text)

    # Complexity
    bigrams  = make_ngrams(tokens, 2)
    trigrams = make_ngrams(tokens, 3)
    word_lengths = [len(w) for w in tokens] or [0]
    # ปรับ: ใช้ try-except สำหรับ pstdev เมื่อมีคำน้อยเกินไป
    try:
        word_len_var = pstdev(word_lengths)
    except:
        word_len_var = 0.0

    # Stylistic
    politeness_cnt   = count_markers(tokens, POLITENESS)
    aggressive_cnt   = count_markers(tokens, AGGRESSIVE)
    urgency_cnt      = count_markers(tokens, URGENCY)
    conditional_cnt  = count_markers(tokens, CONDITIONAL)
    personal_token_cnt = count_markers(tokens, ["you", "your"])
    personal_tag_cnt   = count_phrase_occurrences(email_text, PERSONAL_TAGS)
    personalisation_cnt = personal_token_cnt + personal_tag_cnt

    return {
        # Complexity
        "bigram_total_count": len(bigrams),
        "bigram_unique_count": len(set(bigrams)),
        "trigram_total_count": len(trigrams),
        "trigram_unique_count": len(set(trigrams)),
        "word_length_variation_std": float(word_len_var),
        # Stylistic
        "politeness_markers_count": politeness_cnt,
        "aggressiveness_markers_count": aggressive_cnt,
        "urgency_markers_count": urgency_cnt,
        "conditional_phrases_count": conditional_cnt,
        "personalisation_markers_count": personalisation_cnt,
    }

def analyze_email_body(text):
    """
    คำนวณคุณลักษณะทางภาษาทั้งหมดจากข้อความอีเมล.
    """
    if pd.isna(text) or text is None:
        return (0,) * len(LINGUISTIC_COLUMNS)

    # 1. การเตรียมการ (Tokenization)
    original_text = str(text) # เก็บข้อความดั้งเดิมไว้สำหรับบางการนับ

    # ใช้วิธี tokenize ของ NLTK สำหรับคำและประโยค
    words = word_tokenize(original_text) # ใช้สำหรับการนับเครื่องหมายวรรคตอนและ POS Tag
    lower_words = [word.lower() for word in words if word.isalnum()] # กรองเอาเฉพาะคำที่เป็นตัวอักษร/ตัวเลข
    sentences = sent_tokenize(original_text)

    # คำนวณเบื้องต้น
    word_count = len(lower_words)
    char_count = len(original_text)
    sentence_count = len(sentences)

    # จัดการกรณีที่ word_count เป็น 0 เพื่อป้องกันการหารด้วยศูนย์
    if word_count == 0:
        return (0,) * len(LINGUISTIC_COLUMNS)

    # --- 1-7. Word Counts, Lengths, and Diversity ---
    avg_word_length = sum(len(word) for word in lower_words) / word_count
    avg_sentence_length = word_count / sentence_count if sentence_count > 0 else 0
    unique_word_count = len(set(lower_words))
    lexical_diversity = unique_word_count / word_count

    # --- 8-10. Email, Uppercase Counts ---
    email_count = sum(1 for word in words if '@' in word)

    # Uppercase Word Count (นับคำที่เป็นตัวพิมพ์ใหญ่ทั้งหมดในคำเดิมและเป็นตัวอักษรเท่านั้น)
    upper_count = sum(1 for word in words if word.isupper() and word.isalpha())
    upper_ratio = upper_count / word_count

    # --- 11-12. Complexity Measures ---
    complex_count = sum(1 for word in lower_words if len(word) > 6)

    # Average Syllables per Word (ใช้ textstat)
    try:
        total_syllables = sum(textstat.syllable_count(word) for word in lower_words)
        avg_syllables = total_syllables / word_count
    except:
        avg_syllables = 0


    # --- 13-14. Punctuation Counts ---
    comma_count = original_text.count(',')
    semicolon_count = original_text.count(';')
    colon_count = original_text.count(':')
    exclamation_count = original_text.count('!')
    quotation_count = original_text.count('"')
    dash_count = original_text.count('-')


    # --- 15-19. Density and Ratio Measures ---
    linking_word_count = sum(1 for word in lower_words if word in LINKING_WORDS)
    sentence_complexity_ratio = linking_word_count / word_count

    clause_density = linking_word_count / sentence_count if sentence_count > 0 else 0

    # Pronoun Density (NLTK POS Tagging)
    tagged_words = pos_tag(words) # ใช้คำที่ไม่ได้แปลงเป็นตัวเล็กในการ Tag
    pronoun_count = sum(1 for word, tag in tagged_words if tag in PRONOUN_TAGS)
    pronoun_density = pronoun_count / word_count

    # Preposition Density
    preposition_count = sum(1 for word in lower_words if word in PREPOSITIONS)
    preposition_density = preposition_count / word_count

    # Function Word Density
    function_word_count = sum(1 for word in lower_words if word in FUNCTION_WORDS)
    function_word_density = function_word_count / word_count


    # Return all 23 values
    return (
        word_count, char_count, avg_word_length, sentence_count, avg_sentence_length,
        unique_word_count, lexical_diversity, email_count, upper_count, upper_ratio,
        complex_count, avg_syllables,
        comma_count, semicolon_count, colon_count, exclamation_count, quotation_count, dash_count,
        sentence_complexity_ratio, clause_density, pronoun_density, preposition_density, function_word_density
    )

# =========================
# NEW: BUILD FEATURE DATAFRAME
# =========================

def build_feature_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    รวมการดึงคุณลักษณะทั้งหมดจาก Subject และ Body แล้วส่งคืนเป็น DataFrame
    """
    # 1. รวม Subject + Body เป็นข้อความเดียว
    combined_text = df["Subject"].fillna("") + " " + df["Body"].fillna("")

    # 2. ดึงคุณลักษณะด้าน Stylistic/Complexity
    stylo_features_df = combined_text.apply(extract_stylo_features).apply(pd.Series)

    # 3. ดึงคุณลักษณะด้าน Linguistic
    # เราใช้ Body เป็นหลักตามโค้ดเดิม (df["Body"] ใน analyze_email_body)
    ling_features_series = df["Body"].apply(lambda x: pd.Series(analyze_email_body(x), index=LINGUISTIC_COLUMNS))

    # 4. รวมคุณลักษณะทั้งหมดเข้าด้วยกัน
    features_df = pd.concat([stylo_features_df, ling_features_series], axis=1)

    # 5. รวมกับคอลัมน์ Subject, Body, Label เพื่อให้ Feature DataFrame ครบถ้วน
    result_df = pd.concat([df.reset_index(drop=True), features_df], axis=1)

    return result_df


# =========================
# MAIN: read → combine → featurize → save
# =========================
if __name__ == "__main__":

    # 1. โหลดข้อมูล
    df = load_email_excel(INPUT_PATH) if IS_EXCEL else load_email_csv(INPUT_PATH)

    # 2. สร้างฟีเจอร์จาก Subject + Body
    feature_df = build_feature_dataframe(df)

    # 3. เตรียม X และ y ตามที่โค้ดเดิมตั้งใจไว้
    # เลือกเฉพาะฟีเจอร์เป็น X
    feature_cols = [col for col in feature_df.columns
                    if col not in ["Subject", "Body", "Label"]]

    X = feature_df[feature_cols]
    y = feature_df["Label"]

    # 4. บันทึก Feature DataFrame ที่รวมแล้ว (ตามส่วน MAIN เดิมที่ถูกคอมเมนต์ไว้)
    out = feature_df.copy()
    out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    # 5. แสดงสรุป
    print("--- การทำงานเสร็จสมบูรณ์ ---")
    print(f"บันทึกไฟล์ Feature DataFrame แล้ว → {OUTPUT_CSV}")
    print("\n## ตัวอย่าง Feature DataFrame ที่สร้างขึ้น:")
    print(out[feature_cols + ["Label"]].head())

    print("\n## สรุปข้อมูล X (Feature Matrix):")
    print(f"ขนาด: {X.shape}")
    print(f"คอลัมน์: {X.columns.tolist()}")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


--- การทำงานเสร็จสมบูรณ์ ---
บันทึกไฟล์ Feature DataFrame แล้ว → features_output.csv

## ตัวอย่าง Feature DataFrame ที่สร้างขึ้น:
   bigram_total_count  bigram_unique_count  trigram_total_count  \
0               349.0                323.0                348.0   
1               307.0                282.0                306.0   
2               289.0                265.0                288.0   
3               335.0                320.0                334.0   
4               326.0                313.0                325.0   

   trigram_unique_count  word_length_variation_std  politeness_markers_count  \
0                 346.0                   3.081512                       5.0   
1                 299.0                   2.603052                       4.0   
2                 282.0                   2.833333                       4.0   
3                 333.0                   3.069484                       2.0   
4                 324.0                   3.006575                 